In [ ]:
import re
import unicodedata
import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import geobr
from IPython.display import display

CSV_CASOS_ANO_DIAGNOSTICO  = '/home/anna/tcc/Tratamento/Casos/incidencia _ano_diagnostico.csv'
CSV_CASOS_ESCOLARIDADE     = '/home/anna/tcc/Tratamento/Casos/incidencia _escolaridade.csv'
CSV_CASOS_RACA_COR         = '/home/anna/tcc/Tratamento/Casos/incidencia _raca_cor.csv'
CSV_CASOS_FAIXA_ETARIA     = '/home/anna/tcc/Tratamento/Casos/incidencia _faixa_etaria.csv'
CSV_POPULACAO_FAIXA_ETARIA = '/home/anna/tcc/Tratamento/População - Faixa Etária/tabela_populacao_faixa_etaria.csv'

MUNICIPIO             = None
CODIGO_MUNICIPIO      = None
REGIONAL_SAUDE        = None
ANOS_RANGE_CASOS      = (1985, 2023)
TOP_N_CATEGORIAS      = 25
COR_BARRAS            = "#6ca8e5"
COR_PONTOS            = "#d73f3f"
COR_TEXTO_BARRA       = "#1a1a1a"
LARGURA_GRAFICO       = 2300
ALTURA_GRAFICO_CASOS  = 1700
X_MIN                 = 0
DIVISOR_MEDIA_ANUAL_CASOS       = 10
ANO_REFERENCIA_POPULACAO_CASOS  = 2022
ANO_MALHA_MUNICIPIOS_MG         = 2020
FILTRO_FAIXA_ETARIA             = None
ORDENAR_TABELA_POR              = 'valor_absoluto_casos'

# ── Grafia correta ─────────────────────────────────────────────────────────────
CORRECAO_GRAFIA = {
    'indigena':                     'Indígena',
    'parda':                        'Parda',
    'preta':                        'Preta',
    'branca':                       'Branca',
    'amarela':                      'Amarela',
    'ignorado':                     'Ignorado',
    'ignorada':                     'Ignorado',
    'sem informacao':               'Sem informação',
    'nenhuma':                      'Nenhuma',
    'fundamental incompleto':       'Fundamental incompleto',
    'fundamental completo':         'Fundamental completo',
    'medio incompleto':             'Médio incompleto',
    'medio completo':               'Médio completo',
    'nivel medio':                  'Nível médio',
    'nivel medio incompleto':       'Nível médio incompleto',
    'nivel medio completo':         'Nível médio completo',
    'nivel superior incompleto':    'Nível superior incompleto',
    'nivel superior completo':      'Nível superior completo',
    'nao se aplica':                'Não se aplica',
    'nao informado':                'Não informado',
    'nao informada':                'Não informado',
}

def _corrigir_grafia(texto: str) -> str:
    chave = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode('ascii').lower().strip()
    return CORRECAO_GRAFIA.get(chave, str(texto).strip())

# ── Utilitários ────────────────────────────────────────────────────────────────
def _norm_txt(txt: str) -> str:
    return unicodedata.normalize('NFKD', str(txt)).encode('ascii', 'ignore').decode('ascii').lower().strip()

def _norm_col(col: str) -> str:
    return re.sub(r'\s+', '_', _norm_txt(col))

def _find_first_col(df: pd.DataFrame, candidates_norm: set):
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None

def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    cols_drop = [c for c in df.columns if _norm_col(c).startswith('unnamed')]
    return df.drop(columns=cols_drop) if cols_drop else df.copy()

def _fmt_int(n):
    try:
        return f'{int(n):,}'.replace(',', '.')
    except Exception:
        return str(n)

FAIXAS_ETARIAS_ORDEM = ['0-9','10-19','20-29','30-39','40-49','50-59','60-69','70 ou mais']

def normalizar_faixa_etaria(categoria):
    cat = _norm_txt(categoria)
    if cat in {'70-79','70 a 79','80+','80 ou mais','80 ou mais.','70 ou mais'}:
        return '70 ou mais'
    return str(categoria).strip()

def _normalizar_codigo_municipio(serie: pd.Series) -> pd.Series:
    return (pd.to_numeric(serie, errors='coerce')
              .astype('Int64').astype(str)
              .str.replace('<NA>', '', regex=False)
              .str.zfill(7))

def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    out = _drop_index_cols(df)
    col_mun = _find_first_col(out, {'municipios','municipio','cidades','cidade'})
    col_cod = _find_first_col(out, {'codigo','cod'})
    col_reg = _find_first_col(out, {'regional_de_saude','regional_saude','regional'})
    if MUNICIPIO is not None and col_mun:
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]
    if CODIGO_MUNICIPIO is not None and col_cod:
        out = out[pd.to_numeric(out[col_cod], errors='coerce').astype('Int64') == int(CODIGO_MUNICIPIO)]
    if REGIONAL_SAUDE is not None and col_reg:
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]
    return out

def _year_cols(df):
    cols = []
    for c in df.columns:
        try:
            y = int(c)
            if 1900 <= y <= 2100:
                cols.append(c)
        except:
            pass
    return sorted(cols)

def preparar_ano_diagnostico(df):
    ycols = _year_cols(df)
    s = df[ycols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=0)
    s = s[s > 0]
    return (pd.DataFrame({'ano': s.index.astype(int), 'categoria': s.index.astype(str), 'total': s.values})
              .sort_values('ano').reset_index(drop=True))

def preparar_long_wide(df):
    rows = []
    for c in df.columns:
        txt = str(c)
        if '_' not in txt:
            continue
        ano_str, categoria = txt.rsplit('_', 1)
        try:
            ano = int(ano_str)
        except:
            continue
        categoria = normalizar_faixa_etaria(categoria)
        total = pd.to_numeric(df[c], errors='coerce').fillna(0).sum()
        if total > 0:
            rows.append({'ano': ano, 'categoria': categoria, 'total': int(total)})
    out = pd.DataFrame(rows)
    return out.sort_values(['ano', 'categoria']).reset_index(drop=True)

def build_por_categoria_por_ano(df_long, anos_range=None, top_n=None):
    d = df_long.copy()
    d['ano']   = pd.to_numeric(d['ano'],   errors='coerce').astype('Int64')
    d['total'] = pd.to_numeric(d['total'], errors='coerce')
    d = d.dropna(subset=['ano', 'categoria', 'total'])
    d['ano'] = d['ano'].astype(int)
    d = d[d['total'] > 0]
    if anos_range is not None:
        ai, af = map(int, anos_range)
        d = d[(d['ano'] >= ai) & (d['ano'] <= af)]
    por_categoria = d.groupby('categoria')['total'].sum()
    if top_n is not None:
        por_categoria = por_categoria.sort_values(ascending=False).head(int(top_n)).sort_values()
    por_ano = d.groupby('ano')['total'].sum().sort_index()
    return por_categoria, por_ano

def _carregar_e_preparar_casos(caminho_csv, nome, modo='wide', top_n=None, anos_range=ANOS_RANGE_CASOS):
    df   = aplicar_filtros(pd.read_csv(caminho_csv))
    print(f'Linhas selecionadas ({nome}): {len(df)}')
    long = preparar_ano_diagnostico(df) if modo == 'ano_cols' else preparar_long_wide(df)
    return build_por_categoria_por_ano(long, anos_range=anos_range, top_n=top_n)

print('=== CONFIG / FILTRO ===')
print('MUNICIPIO:',        MUNICIPIO)
print('CODIGO_MUNICIPIO:', CODIGO_MUNICIPIO)
print('REGIONAL_SAUDE:',   REGIONAL_SAUDE)

# ── Parâmetros de eixo ─────────────────────────────────────────────────────────
# Folga extra no range X para os rótulos dos pontos não colidirem com o eixo Y2
# Equivale a ~18% do xmax — os rótulos ficam nessa faixa invisível à direita
FOLGA_ROTULOS_PONTOS = 0.18
_LIMIAR_FORA = 0.10

def _round_up_nice(v):
    v = float(v)
    if v <= 0:
        return 1000
    step = 200 if v <= 2000 else 500 if v <= 10000 else 1000
    return int(((v + step - 1) // step) * step)

def _build_ticks(xmax, n_ticks_max=10):
    raw_step  = xmax / n_ticks_max
    magnitude = 10 ** (len(str(int(raw_step))) - 1)
    step      = max(magnitude, round(raw_step / magnitude) * magnitude)
    return list(range(0, int(xmax) + step, step))

def _resolve_xmax(vals, fixo=None, folga=0.45, extra=0):
    d    = max(vals) if vals else 0
    r    = _round_up_nice(d * (1 + folga))
    base = r if fixo is None else max(int(fixo), r)
    return base + extra

# ── Plot: ano diagnóstico ──────────────────────────────────────────────────────
def plot_ano(por_ano, titulo, xmax_fixo=8000):
    s    = pd.to_numeric(por_ano, errors='coerce').fillna(0)
    s    = s[s > 0]
    anos = s.index.astype(int).tolist()
    vals = s.values.astype(int).tolist()

    xmax  = 8000
    ticks = _build_ticks(xmax, n_ticks_max=8)

    text_pos = 'outside'

    fig = go.Figure(go.Bar(
        x=[str(a) for a in anos],
        y=vals,
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in vals],
        textposition=text_pos,
        textangle=0,
        insidetextanchor='end',
        textfont=dict(size=16, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
    ))
    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',
        margin=dict(l=100, r=200, t=90, b=100),
        xaxis=dict(
            title='Ano do diagnóstico',
            title_font=dict(size=26),
            tickfont=dict(size=18),
            tickangle=0,
            tickmode='array',
            tickvals=[str(a) for a in anos],
            ticktext=[str(a) for a in anos],
            showgrid=True, gridcolor='#d7dce3', gridwidth=1, automargin=True,
        ),
        yaxis=dict(
            title='Número de casos',
            title_font=dict(size=28),
            tickfont=dict(size=16),
            tickmode='array',
            tickvals=ticks,
            ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN, xmax],
            showgrid=True, gridcolor='#d7dce3', gridwidth=1, automargin=True,
        ),
    )
    fig.show()

# ── Plot: duplo ────────────────────────────────────────────────────────────────
def plot_duplo(por_categoria, por_ano, titulo, ytitle, xmax_fixo):
    c = pd.to_numeric(por_categoria, errors='coerce').fillna(0)
    c = c[c > 0]
    a = pd.to_numeric(por_ano, errors='coerce').fillna(0)
    a = a[a > 0]

    is_faixa = _norm_txt(ytitle) == 'faixa etaria'
    if is_faixa:
        ordem = [f for f in FAIXAS_ETARIAS_ORDEM if f in c.index.astype(str).tolist()]
        c = c.reindex(ordem)

    cats = [_corrigir_grafia(cat) for cat in c.index.astype(str).tolist()]
    v1   = c.values.astype(int).tolist()
    anos = a.index.astype(int).tolist()
    v2   = a.values.astype(int).tolist()

    # xmax da área de plotagem (barras chegam até aqui)
    xmax_plot = int(xmax_fixo) if xmax_fixo is not None else _resolve_xmax(v1, None, folga=0.30)
    
    # range do eixo X estendido para os rótulos dos pontos cabarem ANTES do eixo Y2
    xmax_range = int(xmax_plot * (1 + FOLGA_ROTULOS_PONTOS))
    
    ticks = _build_ticks(xmax_plot, n_ticks_max=10)  # ticks só até xmax_plot

    ano_ini, ano_fim = map(int, ANOS_RANGE_CASOS)
    anos_eixo        = list(range(ano_ini, ano_fim + 1))

    fig = go.Figure()

    fig.add_trace(go.Bar(
        name='Total por categoria',
        x=v1,
        y=cats,
        orientation='h',
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in v1],
        textposition='outside',
        textfont=dict(size=20, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
        yaxis='y',
        xaxis='x',
    ))

    fig.add_trace(go.Scatter(
        name='Total por ano',
        x=v2,
        y=anos,
        mode='markers+text',
        marker=dict(color=COR_PONTOS, size=10),
        text=[_fmt_int(v) for v in v2],
        textfont=dict(color=COR_PONTOS, size=20),
        textposition='middle right',
        cliponaxis=False,
        yaxis='y2',
        xaxis='x',
    ))

    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',
        # margem direita generosa: eixo Y2 + rótulos dos pontos
        margin=dict(l=170, r=500, t=95, b=100),
        xaxis=dict(
            title='Número de casos',
            tickmode='array',
            tickvals=ticks,
            ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN, xmax_range],   # range maior que os ticks → rótulos cabem
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18),
            tickangle=0,
        ),
        yaxis=dict(
            title=ytitle,
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            categoryorder='array',
            categoryarray=cats,
            automargin=True,
            title_font=dict(size=32),
            tickfont=dict(size=22),
        ),
        yaxis2=dict(
            title='Ano',
            overlaying='y',
            side='right',
            tickmode='array',
            tickvals=anos_eixo,
            range=[ano_ini - 2, ano_fim + 2],
            showgrid=False,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18),
        ),
    )
    fig.show()

# ── Carregar e plotar ──────────────────────────────────────────────────────────
graficos_casos = []

por_cat, por_ano = _carregar_e_preparar_casos(
    CSV_CASOS_ANO_DIAGNOSTICO, 'Casos - Ano Diagnóstico',
    modo='ano_cols', top_n=None, anos_range=None,
)
por_ano_ref = por_ano[(por_ano.index >= ANOS_RANGE_CASOS[0]) & (por_ano.index <= ANOS_RANGE_CASOS[1])]
graficos_casos.append({
    'tipo': 'ano', 'titulo': 'Casos - Ano Diagnóstico',
    'ylabel': 'Ano do diagnóstico', 'por_categoria': por_cat,
    'por_ano': por_ano_ref, 'x_max': 8000,
})

for nome, csv, ylabel, xmax in [
    ('Casos - Escolaridade', CSV_CASOS_ESCOLARIDADE, 'Escolaridade', 37000),  # ticks até 37k
    ('Casos - Raça/Cor',     CSV_CASOS_RACA_COR,     'Raça/Cor',     55000),  # ticks até 55k
    ('Casos - Faixa Etária', CSV_CASOS_FAIXA_ETARIA, 'Faixa etária', 33000),
]:
    c, a = _carregar_e_preparar_casos(csv, nome, modo='wide',
                                      top_n=TOP_N_CATEGORIAS,
                                      anos_range=ANOS_RANGE_CASOS)
    graficos_casos.append({
        'tipo': 'duplo', 'titulo': nome,
        'ylabel': ylabel, 'por_categoria': c,
        'por_ano': a, 'x_max': xmax,
    })

print(f'Casos: {len(graficos_casos)} gráficos preparados')

for g in graficos_casos:
    if g['tipo'] == 'ano':
        plot_ano(g['por_ano'], f"{g['titulo']} (total por ano)", g['x_max'])
    else:
        plot_duplo(g['por_categoria'], g['por_ano'], g['titulo'], g['ylabel'], g['x_max'])

print('✅ Gráficos de casos plotados')